# 🔥 Feature Engineering Hacks — Hinglish Guide

Is notebook mein hum 10 powerful feature engineering hacks seekhenge.

Har technique ke saath:
- 👉 **Kya hai** — simple explanation
- 👉 **Kab use karein** — use case
- 👉 **Code** — Hinglish comments ke saath

---

## 📦 Libraries Import Karo

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

# -----------------------------------------------
# Ek sample dataset banate hain practice ke liye
# -----------------------------------------------
np.random.seed(42)

df = pd.DataFrame({
    'age':          np.random.randint(18, 70, 200),
    'income':       np.random.randint(20000, 200000, 200),
    'debt':         np.random.randint(0, 50000, 200),
    'price':        np.random.randint(100, 10000, 200),
    'area':         np.random.randint(500, 5000, 200),
    'sales':        np.random.randint(50, 500, 200),
    'review_count': np.random.randint(0, 100, 200),
    'city':         np.random.choice(['Mumbai', 'Delhi', 'Pune', 'Hyderabad', 'Bangalore'], 200),
    'cabin':        np.random.choice(['A1', 'B2', None, None, 'C3'], 200),
    'description':  np.random.choice([
                        'Great product with discount offer',
                        'Amazing quality item',
                        'Best deal discount available',
                        'Standard product no offer'
                    ], 200),
    'target':       np.random.randint(0, 2, 200),
    'date':         pd.date_range('2023-01-01', periods=200, freq='D')
})

print("Dataset shape:", df.shape)
df.head()

---
## 🎯 Hack 1 — Target Encoding

**Kya hai?**  
Har category ko uski **target variable ki mean value** se replace karo.

**Kab use karein?**  
- Jab column mein bahut saari categories ho (high cardinality)
- Tree-based models (XGBoost, Random Forest) ke saath best kaam karta hai

**Avoid karein jab:**  
- Dataset chhota ho — overfitting ho sakta hai
- Hamesha **train data pe fit** karo, test pe sirf transform karo

In [ ]:
# -----------------------------------------------
# Target Encoding
# Har city ke liye — us city ke saare rows ka
# target column ka average nikaalo
# phir us average se city ko replace karo
# -----------------------------------------------

# groupby city karo aur target ka mean nikalo
# transform isliye use kiya taaki original index pe wapas aa jaye
df['city_target_encoded'] = df.groupby('city')['target'].transform('mean')

# Result dekhte hain — city ke saath encoded value
df[['city', 'city_target_encoded']].drop_duplicates().sort_values('city')

---
## 📊 Hack 2 — Frequency Encoding

**Kya hai?**  
Har category ko uski **frequency (proportion)** se replace karo — kitni baar aayi dataset mein.

**Kab use karein?**  
- Jab category ki popularity/rarity ek signal ho
- Target encoding se safe hai — target leakage nahi hota

**Example:** 'Mumbai' 40% rows mein hai → encoded value = 0.40

In [ ]:
# -----------------------------------------------
# Frequency Encoding
# value_counts(normalize=True) se har category
# ki percentage nikalta hai (0 to 1 ke beech)
# phir map() se original column pe apply karte hain
# -----------------------------------------------

# Har city ki frequency (proportion) nikalo
freq_map = df['city'].value_counts(normalize=True)

# City column ko frequency se replace karo
df['city_freq_encoded'] = df['city'].map(freq_map)

# Dekhte hain kya mila
df[['city', 'city_freq_encoded']].drop_duplicates().sort_values('city_freq_encoded', ascending=False)

---
## 🪣 Hack 3 — Binning / Bucketing

**Kya hai?**  
Continuous numerical column ko **groups (bins)** mein tod do.

**Kab use karein?**  
- Jab exact number important na ho, range important ho
- Noisy data ko smooth karne ke liye

**2 types:**
- `pd.cut()` → **Equal width** bins (apne hisaab se ranges define karo)
- `pd.qcut()` → **Equal frequency** bins (har bin mein equal rows)

In [ ]:
# -----------------------------------------------
# Binning — Age ko groups mein todna
# -----------------------------------------------

# pd.cut() — apne hisaab se ranges define karo
# bins = [0, 18, 35, 60, 100] matlab:
#   0-18   → 'teen'
#   18-35  → 'young'
#   35-60  → 'mid'
#   60-100 → 'senior'
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 100],
    labels=['Teen', 'Young', 'Mid-Age', 'Senior']
)

# pd.qcut() — dataset ko 4 equal parts mein baanto
# q=4 matlab 4 quartiles (Q1, Q2, Q3, Q4)
# har quartile mein approximately equal number of rows honge
df['income_quartile'] = pd.qcut(
    df['income'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Dono columns dekhte hain
df[['age', 'age_group', 'income', 'income_quartile']].head(10)

---
## ⏳ Hack 4 — Lag Features (Time Series ke liye)

**Kya hai?**  
Past ke values ko current row ki feature bana do — **"N din pehle kya tha"**

**Kab use karein?**  
- Time series data mein (sales, stock price, user activity)
- Jab past behavior future predict karne mein help kare

**Important:** Pehle date ke hisaab se sort karo warna galat lag banega!

In [ ]:
# -----------------------------------------------
# Lag Features — Time Series
# -----------------------------------------------

# Pehle date ke hisaab se sort karna ZAROORI hai
# warna lag galat row se value uthayega
df = df.sort_values('date').reset_index(drop=True)

# shift(1) — 1 row upar se value uthao
# matlab aaj ki row ko kal ki sales milegi
df['sales_lag_1day'] = df['sales'].shift(1)

# shift(7) — 7 rows upar se value uthao
# matlab aaj ki row ko 1 hafta pehle ki sales milegi
df['sales_lag_7day'] = df['sales'].shift(7)

# shift(30) — 30 din pehle ki sales
df['sales_lag_30day'] = df['sales'].shift(30)

# Pehle kuch NaN aayenge kyunki unke pehle data nahi hai
df[['date', 'sales', 'sales_lag_1day', 'sales_lag_7day', 'sales_lag_30day']].head(10)

---
## 🪟 Hack 5 — Rolling Window Features

**Kya hai?**  
Ek **sliding window** se last N rows ka mean, sum, std nikalo.

**Kab use karein?**  
- Trend capture karna ho (7-day average)
- Volatility/stability dekhni ho (rolling std)
- Noise smooth karna ho

**Lag vs Rolling:**
- Lag = ek specific din ka value
- Rolling = last N dinon ka average/sum

In [ ]:
# -----------------------------------------------
# Rolling Window Features
# -----------------------------------------------

# rolling(7) — last 7 rows ka window banao
# .mean() — us window ka average nikalo
# matlab: aaj ke liye last 7 din ka average sales
df['sales_roll_mean_7'] = df['sales'].rolling(window=7).mean()

# rolling(7).sum() — last 7 din ka total sales
df['sales_roll_sum_7'] = df['sales'].rolling(window=7).sum()

# rolling(7).std() — last 7 din mein kitna variation tha
# zyada std = unstable sales, kam std = stable
df['sales_roll_std_7'] = df['sales'].rolling(window=7).std()

# 30 din ka rolling average — long-term trend
df['sales_roll_mean_30'] = df['sales'].rolling(window=30).mean()

# Pehli N rows NaN rahenge (window fill nahi hogi)
df[['date', 'sales', 'sales_roll_mean_7', 'sales_roll_std_7', 'sales_roll_mean_30']].head(15)

---
## 📐 Hack 6 — Polynomial Features

**Kya hai?**  
Features ke **squares, cubes aur cross-products** banao — non-linear relationships capture karne ke liye.

**Kab use karein?**  
- Linear models (Linear Regression, Logistic Regression) ko non-linear data pe improve karna ho
- Features ke beech interaction important ho

**Dhyan rakho:**  
- degree badhane se features **bahut zyada** ho jaate hain — overfitting ka risk
- Usually degree=2 kaafi hota hai

In [ ]:
# -----------------------------------------------
# Polynomial Features
# -----------------------------------------------

# Sirf 2 features se shuru karte hain: age aur income
features = df[['age', 'income']]

# degree=2 matlab:
#   original: age, income
#   squared:  age², income²
#   cross:    age × income
# include_bias=False matlab constant term (1) mat add karo
poly = PolynomialFeatures(degree=2, include_bias=False)

# fit_transform — features pe apply karo
poly_array = poly.fit_transform(features)

# Numpy array ko DataFrame mein convert karo
# get_feature_names_out() se column names milte hain
poly_df = pd.DataFrame(
    poly_array,
    columns=poly.get_feature_names_out(['age', 'income'])
)

# Kya kya features bane dekhte hain
print("Nayi features jo bani:")
print(poly_df.columns.tolist())
poly_df.head()

---
## 📉 Hack 7 — Log Transformation

**Kya hai?**  
Skewed (right-heavy) numerical column pe **log lagao** — distribution normal jaisi ho jaati hai.

**Kab use karein?**  
- Price, income, population jaise columns pe — jinmein outliers bahut bade hote hain
- Linear models ke liye almost zaroori hai skewed data pe

**`log1p` kyun?**  
`log(0)` undefined hota hai — `log1p(x) = log(1+x)` isliye zero-safe hai

In [ ]:
# -----------------------------------------------
# Log Transformation
# -----------------------------------------------

# np.log1p — log(1 + x) apply karo
# 1 isliye add karte hain taaki x=0 pe bhi kaam kare
# log(0) = -infinity hota, log(1+0) = 0 hota hai
df['log_price']  = np.log1p(df['price'])
df['log_income'] = np.log1p(df['income'])
df['log_debt']   = np.log1p(df['debt'])

# Before vs After comparison
print("Price — Original Stats:")
print(df['price'].describe().round(2))

print("\nPrice — Log Transformed Stats:")
print(df['log_price'].describe().round(2))

df[['price', 'log_price', 'income', 'log_income']].head(10)

---
## ➗ Hack 8 — Ratio Features

**Kya hai?**  
Do columns ko divide karke **nayi meaningful feature** banao.

**Kab use karein?**  
- Jab domain knowledge ho ki ratio important hai
- Scale-independent comparison karna ho

**Tip:** Denominator mein **+1** add karo — zero division error se bachne ke liye

In [ ]:
# -----------------------------------------------
# Ratio Features
# -----------------------------------------------

# price ko area se divide karo
# matlab: ek sqft ka kitna cost hai
# +1 isliye taaki area = 0 ho toh divide by zero na ho
df['price_per_sqft'] = df['price'] / (df['area'] + 1)

# income ko debt se divide karo
# matlab: income kitna debt se zyada hai
# zyada ratio = financially healthy person
df['income_to_debt_ratio'] = df['income'] / (df['debt'] + 1)

# sales ko review_count se divide karo
# matlab: kitni sales per review ho rahi hai
df['sales_per_review'] = df['sales'] / (df['review_count'] + 1)

# Dekhte hain
df[['price', 'area', 'price_per_sqft',
    'income', 'debt', 'income_to_debt_ratio']].head(10)

---
## 🚩 Hack 9 — Has / Missing Indicator

**Kya hai?**  
Missing ya empty values ko **0/1 binary flag** mein convert karo.

**Kab use karein?**  
- Jab missing data random nahi, meaningful ho (MNAR)
- `cabin` missing = economy class passenger (Titanic jaise dataset mein)
- `review_count = 0` = naya product

**Golden rule:** Missing data ko waste mat karo — signal hai!

In [ ]:
# -----------------------------------------------
# Has / Missing Indicator Features
# -----------------------------------------------

# notna() — jo rows mein cabin ka value hai unhe True karo
# na wali rows False rahegi
# astype(int) — True/False ko 1/0 mein convert karo
df['has_cabin'] = df['cabin'].notna().astype(int)

# review_count > 0 check karo
# matlab: kya is product ka koi review hai?
# 1 = haan review hai, 0 = koi review nahi
df['has_reviews'] = (df['review_count'] > 0).astype(int)

# debt > 0 check karo
# matlab: kya is person pe koi debt hai?
df['has_debt'] = (df['debt'] > 0).astype(int)

# Missing value ka breakdown dekhte hain
print("Cabin — Missing vs Present:")
print(df['has_cabin'].value_counts())

df[['cabin', 'has_cabin', 'review_count', 'has_reviews', 'debt', 'has_debt']].head(10)

---
## 📝 Hack 10 — Text-Based Features

**Kya hai?**  
Raw text columns se **numeric features** nikalo — length, word count, keyword presence etc.

**Kab use karein?**  
- Product description, reviews, comments jaise columns ho
- Full NLP nahi karna, sirf basic signals chahiye

**Examples:**
- Lamba description = detailed product
- 'discount' word hai = promotional item
- Zyada words = zyada information

In [ ]:
# -----------------------------------------------
# Text-Based Features
# -----------------------------------------------

# str.len() — description mein total characters kitne hain
# lamba description = zyada detail = better product listing
df['desc_char_length'] = df['description'].str.len()

# str.split() se words ki list banti hai
# apply(len) se us list ki length = total word count
df['desc_word_count'] = df['description'].str.split().apply(len)

# str.contains() — kya description mein 'discount' word hai?
# case=False — uppercase lowercase dono check karega
# na=False — NaN ko False treat karega
# astype(int) — True/False ko 1/0 banao
df['has_discount_keyword'] = df['description'].str.contains(
    'discount', case=False, na=False
).astype(int)

# 'great' keyword check karo
df['has_great_keyword'] = df['description'].str.contains(
    'great', case=False, na=False
).astype(int)

# Dekhte hain sab features ek saath
df[['description', 'desc_char_length', 'desc_word_count',
    'has_discount_keyword', 'has_great_keyword']].head(10)

---
## ✅ Final Check — Saari Nayi Features Ek Saath


In [ ]:
# -----------------------------------------------
# Saari nayi features jo humne banaayi
# -----------------------------------------------

new_features = [
    'city_target_encoded',    # Hack 1 — Target Encoding
    'city_freq_encoded',      # Hack 2 — Frequency Encoding
    'age_group',              # Hack 3 — Binning (cut)
    'income_quartile',        # Hack 3 — Binning (qcut)
    'sales_lag_1day',         # Hack 4 — Lag Features
    'sales_lag_7day',         # Hack 4 — Lag Features
    'sales_roll_mean_7',      # Hack 5 — Rolling Window
    'sales_roll_std_7',       # Hack 5 — Rolling Window
    'log_price',              # Hack 7 — Log Transform
    'log_income',             # Hack 7 — Log Transform
    'price_per_sqft',         # Hack 8 — Ratio Features
    'income_to_debt_ratio',   # Hack 8 — Ratio Features
    'has_cabin',              # Hack 9 — Missing Indicator
    'has_reviews',            # Hack 9 — Missing Indicator
    'desc_char_length',       # Hack 10 — Text Features
    'has_discount_keyword',   # Hack 10 — Text Features
]

print(f"Total nayi features banayi: {len(new_features)}")
print(f"Final dataset shape: {df[new_features].shape}")
df[new_features].head()

---
## 🧠 Golden Rules — Yaad Rakhna!

| Rule | Explanation |
|---|---|
| **Domain knowledge pehle** | Best features data se nahi, samajh se aate hain |
| **Less is more** | 10 strong features > 100 weak features |
| **Leakage se daro** | Future data kabhi feature mein mat daalo |
| **Train pe fit, test pe transform** | Imputation/encoding hamesha train data pe seekho |
| **Har feature validate karo** | Feature importance ya mutual info se check karo |
| **+1 denominator mein** | Zero division se bachne ke liye |
| **log1p use karo** | log(0) undefined hai, log1p zero-safe hai |